In [ ]:
import pyodbc

class Hotel:

    def __init__(self):

        self.conn = pyodbc.connect(
            "DRIVER={SQL Server};"
            "SERVER=SUDARSHAN\\SQLEXPRESS;"
            "DATABASE=HOTEL;"
            "Trusted_Connection=yes;"
        )

        self.cursor = self.conn.cursor()
        print("Connected successfully!")

        
        self.price = {}
        self.cursor.execute("SELECT menu_name, price FROM dbo.Menu")

        for row in self.cursor.fetchall():
            self.price[row[0].lower()] = row[1]

        print("Menu:", self.price)

        self.orders = []
        self.total = 0

    def show_menu(self):

        print("\n{:<15} {:>10}".format("Menu", "Price"))
        print("-"*26)

        for item, price in self.price.items():
            print("{:<15} {:>10}".format(item.upper(), price))

        print("-"*26)

    def Take_Order(self):

        while True:

            menu = input("Enter Menu (type 'done' to stop): ").lower()

            if menu == "done":
                break

            if menu in self.price:
                qty = int(input("Please Enter Quantity: "))
                price = self.price[menu]
                total_amount = price * qty

                
                self.cursor.execute("""
                    INSERT INTO dbo.Orders(menu_name, quantity, total_amount)
                    VALUES (?, ?, ?)
                """, menu, qty, total_amount)

                self.conn.commit()

                self.orders.append([menu, qty, total_amount])
                self.total += total_amount

                print("Order added & saved to database!")

            else:
                print("Invalid menu name.")

   
    def Deliver_Order(self):
        print("\nYour order is delivering... Please wait.")
        print("Delivered successfully!")

   
    def Print_Bill(self):

        items = []
        items.append("-"*50)
        items.append("|" + "{:^48}".format("WELCOME HOTEL") + "|")
        items.append("-"*50)
        items.append("| {:<3} {:<15} {:<12} {:>13} |".format(
            "Sr", "Menu", "Qty", "Amount"))

        for i, item in enumerate(self.orders, 1):
            items.append("| {:<3} {:<15} {:<12} {:>13.2f} |".format(
                i, item[0].upper(), item[1], item[2]))

        items.append("-"*50)
        items.append("| {:<30} {:>15.2f} |".format("Total", self.total))
        items.append("-"*50)

        print_bill = "\n".join(items)

        print(print_bill)
        return print_bill      


    def download_Bill(self):

        generatebill = self.Print_Bill()

        if generatebill:
            with open("bill.txt", "w") as f:
                f.write(generatebill)

            print("Bill generated successfully")
        

    def close(self):
        self.cursor.close()
        self.conn.close()
        print("Connection closed.")

obj = Hotel()
obj.show_menu()
obj.Take_Order()
obj.Deliver_Order()
obj.Print_Bill()
obj.download_Bill()
obj.close()


Connected successfully!
Menu: {'pizza': Decimal('240.000'), 'pasta': Decimal('140.000'), 'burger': Decimal('200.000'), 'idli sambar': Decimal('350.000'), 'medu wada sambar': Decimal('400.000'), 'tea': Decimal('40.000')}

Menu                 Price
--------------------------
PIZZA              240.000
PASTA              140.000
BURGER             200.000
IDLI SAMBAR        350.000
MEDU WADA SAMBAR    400.000
TEA                 40.000
--------------------------
Order added & saved to database!
Order added & saved to database!

Your order is delivering... Please wait.
Delivered successfully!
Bill generated successfully
Connection closed.
